# Cloud Native Satellite Data Demo 1
* Use pystac_client to search the STAC API at Planetary Computer
* Use xarray to load Sentinel-3 OLCI WFR chlorophyll data
* Use hvplot to visualize chlorophyll-a (CHL-NN) over the Gulf of Tunis

**Future goal:** compare with Copernicus Marine gridded ocean colour (CMEMS L4) — same physical variable, different processing level.

In [ ]:
import pystac_client
import planetary_computer
from rich.table import Table
import hvplot.xarray

# Connect to the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [ ]:
collections = list(catalog.get_all_collections())
collections.sort(key=lambda c: c.id)
table = Table("ID", "Title", title="Planetary Computer collections")
for collection in collections:
    table.add_row(collection.id, collection.title)
table

In [ ]:
# Area of interest — Gulf of Tunis
bbox = [10.0, 36.6, 10.9, 37.2]

In [ ]:
search = catalog.search(
    collections=["sentinel-3-olci-wfr-l2-netcdf"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-21",
)

items = search.item_collection()
print(f"Found {len(items)} Sentinel-3 SRAL WAT passes.")

if items:
    first_item = items[0]
    print(f"\nFirst pass: {first_item.datetime}")
    print("\nAssets:")
    for asset_key, asset in first_item.assets.items():
        print(f"- {asset_key}: {asset.title}")

In [ ]:
import xarray as xr
import fsspec

# Take the first item
item = items[0]

def open_asset(item, key):
    asset = planetary_computer.sign(item.assets[key])
    return xr.open_dataset(fsspec.open(asset.href).open())

ds = open_asset(item, "chl-nn")
geo = open_asset(item, "geo-coordinates")

# Assign lat/lon from the geo-coordinates file as coordinates
ds = ds.assign_coords(
    latitude=(["rows", "columns"], geo["latitude"].values),
    longitude=(["rows", "columns"], geo["longitude"].values),
)

print(ds)

In [ ]:
import numpy as np

# Load coordinates (lightweight)
lat = ds["latitude"].values
lon = ds["longitude"].values

# Exact spatial clip to bbox
mask = (
    (lat >= bbox[1]) & (lat <= bbox[3]) &
    (lon >= bbox[0]) & (lon <= bbox[2])
)

rows_idx, cols_idx = np.where(mask)

if len(rows_idx) == 0:
    print("No pixels found in bbox — try another item")
else:
    print(f"{len(rows_idx)} pixels found in the Gulf of Tunis")

    row_min, row_max = rows_idx.min(), rows_idx.max() + 1
    col_min, col_max = cols_idx.min(), cols_idx.max() + 1

    chl_subset = ds["CHL_NN"].isel(
        rows=slice(row_min, row_max),
        columns=slice(col_min, col_max),
    ).load()

    lat_subset = ds["latitude"].isel(
        rows=slice(row_min, row_max),
        columns=slice(col_min, col_max),
    ).values

    lon_subset = ds["longitude"].isel(
        rows=slice(row_min, row_max),
        columns=slice(col_min, col_max),
    ).values

    # Keep the bbox mask for exact clipping (not just the rectangular envelope)
    mask_subset = mask[row_min:row_max, col_min:col_max]

    chl_vals = chl_subset.values.copy()
    chl_vals[~mask_subset] = np.nan

    valid = chl_vals[np.isfinite(chl_vals) & (chl_vals > 0)]
    print(f"CHL_NN min: {valid.min():.3f} mg/m³")
    print(f"CHL_NN max: {valid.max():.3f} mg/m³")
    print(f"CHL_NN mean: {valid.mean():.3f} mg/m³")

In [ ]:
import pandas as pd
import hvplot.pandas

df = pd.DataFrame({
    "lat": lat_subset.flatten(),
    "lon": lon_subset.flatten(),
    "chl": chl_vals.flatten(),
})

# Drop masked / fill-value pixels
df = df[df["chl"].notna() & (df["chl"] > 0) & (df["chl"] < 100)]

# Log-scale is standard for chlorophyll
df["log_chl"] = np.log10(df["chl"])

print(f"Valid pixels: {len(df)}")
print(f"CHL_NN min:  {df['chl'].min():.3f} mg/m³")
print(f"CHL_NN max:  {df['chl'].max():.3f} mg/m³")
print(f"CHL_NN mean: {df['chl'].mean():.3f} mg/m³")

plot = df.hvplot.points(
    x="lon", y="lat",
    c="log_chl",
    colormap="YlGn",
    clim=(df["log_chl"].quantile(0.02), df["log_chl"].quantile(0.98)),
    title="Chlorophyll-a (CHL-NN) — Gulf of Tunisia (log₁₀ mg/m³)",
    xlabel="Longitude",
    ylabel="Latitude",
    clabel="log₁₀ CHL (mg/m³)",
    size=6,
    geo=True,
    tiles="OSM",
    width=700,
    height=500,
)

plot